# GaitFuseNet Experiments — Model 1: IMU-only Baseline

This is the first rung of the experimental ladder:

| # | Model | IMU | Insole | Fusion |
|---|---|---|---|---|
| **1** | **IMU-only** | ✓ | — | — |
| 2 | Insole-only | — | ✓ | — |
| 3 | Naive late fusion | ✓ | ✓ | Concatenation |
| 4 | GaitFuseNet | ✓ | ✓ | Cross-modal attention |
| 5 | GaitFuseNet + temporal attention | ✓ | ✓ | Cross-modal + temporal |

This notebook is structured so Sections 1-4 (data loading, normalization, Dataset/DataLoader, training loop) are reused unchanged for Models 2-5 — only Section 5 (the model definition) changes between experiments.

## 1. Load the saved splits

Add your Kaggle dataset as input first (sidebar → Add Input), then adjust `DATA_DIR` below to match.

In [1]:
import gc
import torch

# Delete model, optimizer, scheduler, and data-related objects
for var_name in ['model', 'optimizer', 'scheduler', 'train_loader', 'val_loader', 'test_loader',
                  'train_ds', 'val_ds', 'test_ds',
                  'train_data', 'val_data', 'test_data',
                  'X_imu_train_norm', 'X_imu_val_norm', 'X_imu_test_norm',
                  'sev_train_scaled', 'sev_val_scaled', 'sev_test_scaled',
                  'train_metrics', 'val_metrics', 'test_metrics', 'history']:
    if var_name in globals():
        del globals()[var_name]

# Force garbage collection
gc.collect()

# Clear GPU memory cache (if using CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

print("RAM/GPU cleared.")

GPU memory allocated: 0.00 GB
GPU memory reserved:  0.00 GB
RAM/GPU cleared.


In [2]:
import numpy as np
import os

DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"  # adjust to match your actual dataset path

def load_split(split_name):
    return {
        'X_imu': np.load(os.path.join(DATA_DIR, f'X_imu_{split_name}.npy')),
        'X_insole': np.load(os.path.join(DATA_DIR, f'X_insole_{split_name}.npy')),
        'y': np.load(os.path.join(DATA_DIR, f'y_{split_name}.npy')),
        'severity_part3': np.load(os.path.join(DATA_DIR, f'severity_part3_{split_name}.npy')),
        'severity_total': np.load(os.path.join(DATA_DIR, f'severity_total_{split_name}.npy')),
        'subject': np.load(os.path.join(DATA_DIR, f'subject_{split_name}.npy'), allow_pickle=True),
    }

train_data = load_split('train')
val_data = load_split('val')
test_data = load_split('test')

for name, d in [('Train', train_data), ('Val', val_data), ('Test', test_data)]:
    print(f"{name}: X_imu {d['X_imu'].shape} | y {d['y'].shape} | "
          f"subjects {len(np.unique(d['subject']))} | class balance {np.bincount(d['y'].astype(int))}")

Train: X_imu (17066, 300, 286) | y (17066,) | subjects 118 | class balance [7354 9712]
Val: X_imu (4006, 300, 286) | y (4006,) | subjects 25 | class balance [1747 2259]
Test: X_imu (3493, 300, 286) | y (3493,) | subjects 26 | class balance [1630 1863]


In [4]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# 2. LOAD FINAL SPLIT ARRAYS AS MEMORY-MAPPED ARRAYS
# ============================================================

# IMU arrays stay on disk; they are NOT loaded fully into RAM
X_imu_train = np.load(
    f"{DATA_DIR}/X_imu_train.npy",
    mmap_mode="r"
)

X_imu_val = np.load(
    f"{DATA_DIR}/X_imu_val.npy",
    mmap_mode="r"
)

X_imu_test = np.load(
    f"{DATA_DIR}/X_imu_test.npy",
    mmap_mode="r"
)

# Small arrays can safely be loaded normally
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")

sev_train = np.load(
    f"{DATA_DIR}/severity_part3_train.npy"
)

sev_val = np.load(
    f"{DATA_DIR}/severity_part3_val.npy"
)

sev_test = np.load(
    f"{DATA_DIR}/severity_part3_test.npy"
)

print("Train IMU:", X_imu_train.shape)
print("Val IMU:  ", X_imu_val.shape)
print("Test IMU: ", X_imu_test.shape)


# ============================================================
# 3. COMPUTE NORMALIZATION FROM TRAINING DATA ONLY
# ============================================================

# One mean/std per IMU feature.
# Calculated across:
#   all training windows × all 300 time steps

imu_mean = np.asarray(
    X_imu_train.mean(axis=(0, 1)),
    dtype=np.float32
)

imu_std = np.asarray(
    X_imu_train.std(axis=(0, 1)),
    dtype=np.float32
)

# Prevent division by zero
imu_std[imu_std < 1e-6] = 1.0

print("IMU mean shape:", imu_mean.shape)
print("IMU std shape:", imu_std.shape)


# ============================================================
# 4. SCALE SEVERITY USING TRAINING DATA ONLY
# ============================================================

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())

if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (
    sev_train - severity_mean
) / severity_std

sev_val_scaled = (
    sev_val - severity_mean
) / severity_std

sev_test_scaled = (
    sev_test - severity_mean
) / severity_std

print("Severity mean:", severity_mean)
print("Severity std:", severity_std)


# ============================================================
# 5. MEMORY-SAFE DATASET
# ============================================================

class GaitDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        severity,
        mean,
        std,
        augment=False,
        noise_std=0.05
    ):

        # IMPORTANT:
        # X remains a NumPy memmap.
        # We do NOT convert the entire dataset to torch.
        self.X = X

        self.y = y
        self.severity = severity

        # Store normalization statistics
        self.mean = mean
        self.std = std

        # Augmentation settings
        self.augment = augment
        self.noise_std = noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

        # ----------------------------------------------------
        # Load ONLY ONE 300 × 286 window from disk
        # ----------------------------------------------------

        x = np.asarray(
            self.X[idx],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Normalize using TRAINING statistics
        # ----------------------------------------------------

        x = (x - self.mean) / self.std

        # Convert only this window to a tensor
        x = torch.from_numpy(x)

        # ----------------------------------------------------
        # DATA AUGMENTATION
        # ----------------------------------------------------
        # Applied ONLY when augment=True.
        #
        # Training:
        #     augment=True
        #
        # Validation:
        #     augment=False
        #
        # Test:
        #     augment=False
        # ----------------------------------------------------

        if self.augment:
            noise = torch.randn_like(x) * self.noise_std
            x = x + noise

        # ----------------------------------------------------
        # Targets
        # ----------------------------------------------------

        y = torch.tensor(
            self.y[idx],
            dtype=torch.float32
        )

        severity = torch.tensor(
            self.severity[idx],
            dtype=torch.float32
        )

        return x, y, severity


# ============================================================
# 6. CREATE DATASETS
# ============================================================

train_ds = GaitDataset(
    X=X_imu_train,
    y=y_train,
    severity=sev_train_scaled,
    mean=imu_mean,
    std=imu_std,
    augment=True,
    noise_std=0.05
)

val_ds = GaitDataset(
    X=X_imu_val,
    y=y_val,
    severity=sev_val_scaled,
    mean=imu_mean,
    std=imu_std,
    augment=False
)

test_ds = GaitDataset(
    X=X_imu_test,
    y=y_test,
    severity=sev_test_scaled,
    mean=imu_mean,
    std=imu_std,
    augment=False
)


# ============================================================
# 7. DATALOADERS
# ============================================================

BATCH_SIZE = 64

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


# ============================================================
# 8. CLASS WEIGHT
# ============================================================

n_class0 = np.sum(y_train == 0)
n_class1 = np.sum(y_train == 1)

pos_weight = torch.tensor(
    [n_class0 / n_class1],
    dtype=torch.float32,
    device=device
)

print(f"Class 0 (Control): {n_class0}")
print(f"Class 1 (PD):      {n_class1}")
print(f"pos_weight:        {pos_weight.item():.3f}")


# ============================================================
# 9. FINAL CHECK
# ============================================================

print("\nDataset sizes:")
print("Train:", len(train_ds))
print("Val:  ", len(val_ds))
print("Test: ", len(test_ds))

print("\nOne sample:")
sample_x, sample_y, sample_sev = train_ds[0]

print("X shape:", sample_x.shape)
print("X dtype:", sample_x.dtype)
print("y:", sample_y.item())
print("severity:", sample_sev.item())

Using device: cuda
Train IMU: (17066, 300, 286)
Val IMU:   (4006, 300, 286)
Test IMU:  (3493, 300, 286)
IMU mean shape: (286,)
IMU std shape: (286,)
Severity mean: 18.17444
Severity std: 13.93664
Class 0 (Control): 7354
Class 1 (PD):      9712
pos_weight:        0.757

Dataset sizes:
Train: 17066
Val:   4006
Test:  3493

One sample:
X shape: torch.Size([300, 286])
X dtype: torch.float32
y: 0.0
severity: -0.5147898197174072


In [7]:
# ============================================================
# EVALUATE EXISTING IMU-ONLY CHECKPOINT ON TEST SET
# ============================================================

import torch
import numpy as np
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ------------------------------------------------------------
# 2. RECREATE EXACT MODEL ARCHITECTURE
# ------------------------------------------------------------

class IMUOnlyModel(nn.Module):

    def __init__(
        self,
        n_features,
        cnn_channels=32,
        lstm_hidden=64,
        dropout=0.5
    ):

        super().__init__()

        self.conv1 = nn.Conv1d(
            n_features,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn1 = nn.BatchNorm1d(
            cnn_channels
        )

        self.conv2 = nn.Conv1d(
            cnn_channels,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            cnn_channels
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(2)

        self.dropout_conv = nn.Dropout(
            dropout * 0.5
        )

        self.bilstm = nn.LSTM(
            cnn_channels,
            lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        shared_dim = lstm_hidden * 2

        self.shared = nn.Sequential(
            nn.Linear(
                shared_dim,
                64
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(
            64,
            1
        )

        self.regression_head = nn.Linear(
            64,
            1
        )


    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = self.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)

        h_cat = torch.cat(
            [
                h_n[0],
                h_n[1]
            ],
            dim=1
        )

        shared_features = self.shared(
            h_cat
        )

        cls_logit = self.classification_head(
            shared_features
        ).squeeze(-1)

        reg_output = self.regression_head(
            shared_features
        ).squeeze(-1)

        return cls_logit, reg_output


# ------------------------------------------------------------
# 3. CREATE MODEL
# ------------------------------------------------------------

n_imu_features = X_imu_test.shape[2]

model = IMUOnlyModel(
    n_features=n_imu_features,
    cnn_channels=32,
    lstm_hidden=64,
    dropout=0.5
).to(device)


print(
    f"Total parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ------------------------------------------------------------
# 4. LOAD TRAINED CHECKPOINT
# ------------------------------------------------------------

checkpoint_path = (
    "/kaggle/input/models/ananyakukreja06/imuonly/pytorch/default/1/model1_imu_only_best (1).pt"
)

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

print("\nCheckpoint loaded successfully:")
print(checkpoint_path)


# ------------------------------------------------------------
# 5. EVALUATION-ONLY MODE
# ------------------------------------------------------------

model.eval()

all_preds = []
all_labels = []

all_reg_preds = []
all_reg_true = []


with torch.no_grad():

    for xb, yb, sevb in test_loader:

        xb = xb.to(device)
        yb = yb.to(device)
        sevb = sevb.to(device)

        # Forward pass ONLY
        cls_logit, reg_out = model(xb)

        # Classification probability
        probs = torch.sigmoid(
            cls_logit
        )

        all_preds.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            yb.cpu().numpy()
        )

        # Regression
        all_reg_preds.extend(
            reg_out.cpu().numpy()
        )

        all_reg_true.extend(
            sevb.cpu().numpy()
        )


# Convert to numpy
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

all_reg_preds = np.array(all_reg_preds)
all_reg_true = np.array(all_reg_true)


# ------------------------------------------------------------
# 6. CLASSIFICATION METRICS
# ------------------------------------------------------------

pred_labels = (
    all_preds > 0.5
).astype(int)


accuracy = accuracy_score(
    all_labels,
    pred_labels
)

precision = precision_score(
    all_labels,
    pred_labels,
    zero_division=0
)

recall = recall_score(
    all_labels,
    pred_labels,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    pred_labels,
    zero_division=0
)

auroc = roc_auc_score(
    all_labels,
    all_preds
)


# ------------------------------------------------------------
# 7. REGRESSION METRICS
# ------------------------------------------------------------

# Convert scaled predictions back
true_severity = (
    all_reg_true * severity_std
    + severity_mean
)

pred_severity = (
    all_reg_preds * severity_std
    + severity_mean
)


mae = np.mean(
    np.abs(
        true_severity - pred_severity
    )
)

rmse = np.sqrt(
    np.mean(
        (true_severity - pred_severity) ** 2
    )
)


# ------------------------------------------------------------
# 8. FINAL TEST RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)

print(
    "MODEL 1 (IMU-ONLY, REGULARIZED)"
)

print(
    "TEST RESULTS — EVALUATION ONLY"
)

print("=" * 65)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1       : {f1:.4f}"
)

print(
    f"AUROC    : {auroc:.4f}"
)

print(
    f"MAE      : {mae:.3f}"
)

print(
    f"RMSE     : {rmse:.3f}"
)

print("=" * 65)

print(
    f"\nTest samples evaluated: "
    f"{len(all_labels)}"
)

Using device: cuda
Total parameters: 109,634

Checkpoint loaded successfully:
/kaggle/input/models/ananyakukreja06/imuonly/pytorch/default/1/model1_imu_only_best (1).pt

MODEL 1 (IMU-ONLY, REGULARIZED)
TEST RESULTS — EVALUATION ONLY
Accuracy : 0.8674
Precision: 0.8680
Recall   : 0.8862
F1       : 0.8770
AUROC    : 0.9496
MAE      : 9.261
RMSE     : 12.040

Test samples evaluated: 3493


In [9]:
import pandas as pd
from sklearn.metrics import roc_auc_score

# Load test subject IDs
subj_test = np.load(
    f"{DATA_DIR}/subject_test.npy",
    allow_pickle=True
)

# Build window-level dataframe
df_imu = pd.DataFrame({
    "subject": subj_test,
    "true_label": all_labels,
    "pred_prob": all_preds
})

# Aggregate predictions per subject
imu_subject_level = (
    df_imu
    .groupby("subject")
    .agg(
        true_label=("true_label", "first"),
        mean_pred_prob=("pred_prob", "mean")
    )
    .reset_index()
)

# Subject-level AUROC
imu_subject_auroc = roc_auc_score(
    imu_subject_level["true_label"],
    imu_subject_level["mean_pred_prob"]
)

print(
    f"Model 1 (IMU-only) subject-level AUROC: "
    f"{imu_subject_auroc:.4f} "
    f"(n={len(imu_subject_level)})"
)

Model 1 (IMU-only) subject-level AUROC: 0.9702 (n=26)


In [5]:
model.load_state_dict(torch.load('/kaggle/input/models/ananyakukreja06/imuonly/pytorch/default/1/model1_imu_only_best (1).pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

print("MODEL 2 (INSOLE-ONLY) — TEST RESULTS")
print(f"Accuracy: {test_cls['accuracy']:.4f} | Precision: {test_cls['precision']:.4f} | "
      f"Recall: {test_cls['recall']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {'model': 'Model 2 - Insole only', 'classification': test_cls, 'regression': test_reg}
with open('/kaggle/working/model2_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

NameError: name 'model' is not defined

## 2. Normalize (fit on TRAIN only, apply to all three splits)

This is important: computing mean/std from train data only, then applying those same statistics to val/test, avoids any information leaking from val/test into training.

In [7]:
def fit_normalizer(X_train):
    # X_train shape: (n_windows, time, features) -> compute per-feature mean/std across windows and time
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std[std == 0] = 1.0  # avoid divide-by-zero for constant features
    return mean, std

def apply_normalizer(X, mean, std):
    return (X - mean) / std

imu_mean, imu_std = fit_normalizer(train_data['X_imu'])

X_imu_train_norm = apply_normalizer(train_data['X_imu'], imu_mean, imu_std)
X_imu_val_norm = apply_normalizer(val_data['X_imu'], imu_mean, imu_std)
X_imu_test_norm = apply_normalizer(test_data['X_imu'], imu_mean, imu_std)

print("Normalized. Train mean/std check (should be ~0/~1):")
print("Mean:", X_imu_train_norm.mean(), "| Std:", X_imu_train_norm.std())

# Save normalizer stats — you'll need the SAME mean/std later for Models 3-5 too
np.save('/kaggle/working/imu_norm_mean.npy', imu_mean)
np.save('/kaggle/working/imu_norm_std.npy', imu_std)

Normalized. Train mean/std check (should be ~0/~1):
Mean: -0.004544546 | Std: 1.0043191


## 3. Severity target scaling

Severity is on a different scale than the [0,1] classification output — scale it to a reasonable range using train statistics, so the regression loss doesn't dominate or vanish relative to the classification loss.

In [9]:
sev_mean = train_data['severity_part3'].mean()
sev_std = train_data['severity_part3'].std()
print(f"Severity (Part III) train mean: {sev_mean:.2f}, std: {sev_std:.2f}")

def scale_severity(sev, mean, std):
    return (sev - mean) / std

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

sev_train_scaled = scale_severity(train_data['severity_part3'], sev_mean, sev_std)
sev_val_scaled = scale_severity(val_data['severity_part3'], sev_mean, sev_std)
sev_test_scaled = scale_severity(test_data['severity_part3'], sev_mean, sev_std)

np.save('/kaggle/working/severity_scale_mean.npy', np.array([sev_mean]))
np.save('/kaggle/working/severity_scale_std.npy', np.array([sev_std]))

Severity (Part III) train mean: 18.17, std: 13.94


## 4. Dataset, DataLoader, and class weighting

In [3]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# 2. LOAD FINAL SPLIT ARRAYS AS MEMORY-MAPPED ARRAYS
# ============================================================

# IMU arrays stay on disk; they are NOT loaded fully into RAM
X_imu_train = np.load(
    f"{DATA_DIR}/X_imu_train.npy",
    mmap_mode="r"
)

X_imu_val = np.load(
    f"{DATA_DIR}/X_imu_val.npy",
    mmap_mode="r"
)

X_imu_test = np.load(
    f"{DATA_DIR}/X_imu_test.npy",
    mmap_mode="r"
)

# Small arrays can safely be loaded normally
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")

sev_train = np.load(
    f"{DATA_DIR}/severity_part3_train.npy"
)

sev_val = np.load(
    f"{DATA_DIR}/severity_part3_val.npy"
)

sev_test = np.load(
    f"{DATA_DIR}/severity_part3_test.npy"
)

print("Train IMU:", X_imu_train.shape)
print("Val IMU:  ", X_imu_val.shape)
print("Test IMU: ", X_imu_test.shape)


# ============================================================
# 3. COMPUTE NORMALIZATION FROM TRAINING DATA ONLY
# ============================================================

# One mean/std per IMU feature.
# Calculated across:
#   all training windows × all 300 time steps

imu_mean = np.asarray(
    X_imu_train.mean(axis=(0, 1)),
    dtype=np.float32
)

imu_std = np.asarray(
    X_imu_train.std(axis=(0, 1)),
    dtype=np.float32
)

# Prevent division by zero
imu_std[imu_std < 1e-6] = 1.0

print("IMU mean shape:", imu_mean.shape)
print("IMU std shape:", imu_std.shape)


# ============================================================
# 4. SCALE SEVERITY USING TRAINING DATA ONLY
# ============================================================

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())

if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (
    sev_train - severity_mean
) / severity_std

sev_val_scaled = (
    sev_val - severity_mean
) / severity_std

sev_test_scaled = (
    sev_test - severity_mean
) / severity_std

print("Severity mean:", severity_mean)
print("Severity std:", severity_std)


# ============================================================
# 5. MEMORY-SAFE DATASET
# ============================================================

class GaitDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        severity,
        mean,
        std,
        augment=False,
        noise_std=0.05
    ):

        # IMPORTANT:
        # X remains a NumPy memmap.
        # We do NOT convert the entire dataset to torch.
        self.X = X

        self.y = y
        self.severity = severity

        # Store normalization statistics
        self.mean = mean
        self.std = std

        # Augmentation settings
        self.augment = augment
        self.noise_std = noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

        # ----------------------------------------------------
        # Load ONLY ONE 300 × 286 window from disk
        # ----------------------------------------------------

        x = np.asarray(
            self.X[idx],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Normalize using TRAINING statistics
        # ----------------------------------------------------

        x = (x - self.mean) / self.std

        # Convert only this window to a tensor
        x = torch.from_numpy(x)

        # ----------------------------------------------------
        # DATA AUGMENTATION
        # ----------------------------------------------------
        # Applied ONLY when augment=True.
        #
        # Training:
        #     augment=True
        #
        # Validation:
        #     augment=False
        #
        # Test:
        #     augment=False
        # ----------------------------------------------------

        if self.augment:
            noise = torch.randn_like(x) * self.noise_std
            x = x + noise

        # ----------------------------------------------------
        # Targets
        # ----------------------------------------------------

        y = torch.tensor(
            self.y[idx],
            dtype=torch.float32
        )

        severity = torch.tensor(
            self.severity[idx],
            dtype=torch.float32
        )

        return x, y, severity


# ============================================================
# 6. CREATE DATASETS
# ============================================================

train_ds = GaitDataset(
    X=X_imu_train,
    y=y_train,
    severity=sev_train_scaled,
    mean=imu_mean,
    std=imu_std,
    augment=True,
    noise_std=0.05
)

val_ds = GaitDataset(
    X=X_imu_val,
    y=y_val,
    severity=sev_val_scaled,
    mean=imu_mean,
    std=imu_std,
    augment=False
)

test_ds = GaitDataset(
    X=X_imu_test,
    y=y_test,
    severity=sev_test_scaled,
    mean=imu_mean,
    std=imu_std,
    augment=False
)


# ============================================================
# 7. DATALOADERS
# ============================================================

BATCH_SIZE = 64

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


# ============================================================
# 8. CLASS WEIGHT
# ============================================================

n_class0 = np.sum(y_train == 0)
n_class1 = np.sum(y_train == 1)

pos_weight = torch.tensor(
    [n_class0 / n_class1],
    dtype=torch.float32,
    device=device
)

print(f"Class 0 (Control): {n_class0}")
print(f"Class 1 (PD):      {n_class1}")
print(f"pos_weight:        {pos_weight.item():.3f}")


# ============================================================
# 9. FINAL CHECK
# ============================================================

print("\nDataset sizes:")
print("Train:", len(train_ds))
print("Val:  ", len(val_ds))
print("Test: ", len(test_ds))

print("\nOne sample:")
sample_x, sample_y, sample_sev = train_ds[0]

print("X shape:", sample_x.shape)
print("X dtype:", sample_x.dtype)
print("y:", sample_y.item())
print("severity:", sample_sev.item())

Using device: cuda
Train IMU: (17066, 300, 286)
Val IMU:   (4006, 300, 286)
Test IMU:  (3493, 300, 286)
IMU mean shape: (286,)
IMU std shape: (286,)
Severity mean: 18.17444
Severity std: 13.93664
Class 0 (Control): 7354
Class 1 (PD):      9712
pos_weight:        0.757

Dataset sizes:
Train: 17066
Val:   4006
Test:  3493

One sample:
X shape: torch.Size([300, 286])
X dtype: torch.float32
y: 0.0
severity: -0.5147898197174072


## 5. Model 1: IMU-only 1D-CNN + BiLSTM, multi-task heads

In [4]:
# ============================================================
# 10. MODEL — WITH REGULARIZATION FIXES
# ============================================================

class IMUOnlyModel(nn.Module):
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)  # lighter dropout on conv features

        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)

        shared_dim = lstm_hidden * 2
        self.shared = nn.Sequential(
            nn.Linear(shared_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)

        shared_features = self.shared(h_cat)
        cls_logit = self.classification_head(shared_features).squeeze(-1)
        reg_output = self.regression_head(shared_features).squeeze(-1)
        return cls_logit, reg_output

n_imu_features = X_imu_train.shape[2]
model = IMUOnlyModel(n_imu_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 11. LOSS, OPTIMIZER, TRAINING LOOP
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()

LAMBDA_CLS = 1.0
LAMBDA_REG = 1.0   # raised from 0.5 — forces more balanced attention to both tasks

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # raised from 1e-5
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for xb, yb, sevb in loader:
            xb, yb, sevb = xb.to(device), yb.to(device), sevb.to(device)

            cls_logit, reg_out = model(xb)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            total_cls_loss += cls_loss.item() * xb.size(0)
            total_reg_loss += reg_loss.item() * xb.size(0)

            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}

N_EPOCHS = 30
PATIENCE = 7
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} (cls={train_metrics['cls_loss']:.4f}, reg={train_metrics['reg_loss']:.4f}) "
          f"train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model1_imu_only_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print("\nTraining complete.")


# ============================================================
# 12. TEST EVALUATION
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model1_imu_only_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

print("MODEL 1 (IMU-ONLY, REGULARIZED) — TEST RESULTS")
print(f"Accuracy: {test_cls['accuracy']:.4f} | Precision: {test_cls['precision']:.4f} | "
      f"Recall: {test_cls['recall']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

IMUOnlyModel(
  (conv1): Conv1d(286, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout_conv): Dropout(p=0.25, inplace=False)
  (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  (shared): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
  )
  (classification_head): Linear(in_features=64, out_features=1, bias=True)
  (regression_head): Linear(in_features=64, out_features=1, bias=True)
)

Total parameters: 109,634
Epoch  1/30 | train_loss=0.7737 (cls=0.2243, reg=0.5494) train_acc=0.892 | val_loss=1.6007 val_acc=0.825 val_f1=0.856 val_auroc=0.89

In [5]:
import json
results = {'model': 'Model 1 - IMU only (regularized)', 'classification': test_cls, 'regression': test_reg}
with open('/kaggle/working/model1_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score

# Load test subject IDs
subj_test = np.load(
    f"{DATA_DIR}/subject_test.npy",
    allow_pickle=True
)

# Build window-level dataframe
df_imu = pd.DataFrame({
    "subject": subj_test,
    "true_label": all_labels,
    "pred_prob": all_preds
})

# Aggregate predictions per subject
imu_subject_level = (
    df_imu
    .groupby("subject")
    .agg(
        true_label=("true_label", "first"),
        mean_pred_prob=("pred_prob", "mean")
    )
    .reset_index()
)

# Subject-level AUROC
imu_subject_auroc = roc_auc_score(
    imu_subject_level["true_label"],
    imu_subject_level["mean_pred_prob"]
)

print(
    f"Model 1 (IMU-only) subject-level AUROC: "
    f"{imu_subject_auroc:.4f} "
    f"(n={len(imu_subject_level)})"
)

NameError: name 'np' is not defined

# Model 2: Insole-only 1D-CNN + BiLSTM, multi-task heads

In [12]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# 2. LOAD FINAL SPLIT ARRAYS AS MEMORY-MAPPED ARRAYS
# ============================================================


X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")

y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")

sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")

print("Train Insole:", X_insole_train.shape)
print("Val Insole:  ", X_insole_val.shape)
print("Test Insole: ", X_insole_test.shape)


# ============================================================
# 3. COMPUTE NORMALIZATION FROM TRAINING DATA ONLY
# ============================================================

insole_mean = np.asarray(X_insole_train.mean(axis=(0, 1)), dtype=np.float32)
insole_std = np.asarray(X_insole_train.std(axis=(0, 1)), dtype=np.float32)
insole_std[insole_std < 1e-6] = 1.0

print("Insole mean shape:", insole_mean.shape)
print("Insole std shape:", insole_std.shape)


# ============================================================
# 4. SCALE SEVERITY (reuse same train stats convention)
# ============================================================

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std

print("Severity mean:", severity_mean, "| std:", severity_std)


# ============================================================
# 5. MEMORY-SAFE DATASET (identical structure to Model 1)
# ============================================================

class GaitDatasetPerWindowNorm(Dataset):
    def __init__(self, X, y, severity, augment=False, noise_std=0.05):
        self.X = X
        self.y = y
        self.severity = severity
        self.augment = augment
        self.noise_std = noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        # Normalize THIS window using its own mean/std, per feature — removes person-specific baseline/scale
        w_mean = x.mean(axis=0, keepdims=True)
        w_std = x.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x = (x - w_mean) / w_std

        x = torch.from_numpy(x)
        if self.augment:
            x = x + torch.randn_like(x) * self.noise_std
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        severity = torch.tensor(self.severity[idx], dtype=torch.float32)
        return x, y, severity

train_ds = GaitDatasetPerWindowNorm(X_insole_train, y_train, sev_train_scaled, augment=True, noise_std=0.05)
val_ds = GaitDatasetPerWindowNorm(X_insole_val, y_val, sev_val_scaled, augment=False)
test_ds = GaitDatasetPerWindowNorm(X_insole_test, y_test, sev_test_scaled, augment=False)

# ============================================================
# 6. DATASETS AND DATALOADERS
# ============================================================

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

n_class0 = np.sum(y_train == 0)
n_class1 = np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)
print(f"Class 0 (Control): {n_class0} | Class 1 (PD): {n_class1} | pos_weight: {pos_weight.item():.3f}")


# ============================================================
# 7. MODEL — same architecture as Model 1, just different input width
# ============================================================

class InsoleOnlyModel(nn.Module):
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)

        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)

        shared_dim = lstm_hidden * 2
        self.shared = nn.Sequential(
            nn.Linear(shared_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)

        shared_features = self.shared(h_cat)
        cls_logit = self.classification_head(shared_features).squeeze(-1)
        reg_output = self.regression_head(shared_features).squeeze(-1)
        return cls_logit, reg_output

n_insole_features = X_insole_train.shape[2]
model = InsoleOnlyModel(n_insole_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 8. LOSS, OPTIMIZER, TRAINING LOOP
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()

LAMBDA_CLS = 1.0
LAMBDA_REG = 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for xb, yb, sevb in loader:
            xb, yb, sevb = xb.to(device), yb.to(device), sevb.to(device)
            cls_logit, reg_out = model(xb)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            total_cls_loss += cls_loss.item() * xb.size(0)
            total_reg_loss += reg_loss.item() * xb.size(0)
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}

N_EPOCHS = 30
PATIENCE = 7
best_val_auroc = -float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])  # scheduler can still watch loss, that's fine

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} "
          f"val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_cls['auroc'] > best_val_auroc:   # <-- changed: track AUROC, maximize
        best_val_auroc = val_cls['auroc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model2_insole_only_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val AUROC: {best_val_auroc:.4f}")
# ============================================================
# 9. TEST EVALUATION
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model2_insole_only_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

print("MODEL 2 (INSOLE-ONLY) — TEST RESULTS")
print(f"Accuracy: {test_cls['accuracy']:.4f} | Precision: {test_cls['precision']:.4f} | "
      f"Recall: {test_cls['recall']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {'model': 'Model 2 - Insole only', 'classification': test_cls, 'regression': test_reg}
with open('/kaggle/working/model2_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

Using device: cuda
Train Insole: (17066, 300, 50)
Val Insole:   (4006, 300, 50)
Test Insole:  (3493, 300, 50)
Insole mean shape: (50,)
Insole std shape: (50,)
Severity mean: 18.17444 | std: 13.93664
Class 0 (Control): 7354 | Class 1 (PD): 9712 | pos_weight: 0.757
InsoleOnlyModel(
  (conv1): Conv1d(50, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout_conv): Dropout(p=0.25, inplace=False)
  (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  (shared): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
  )
  (classification_head): Linear(in_featur

In [13]:
import pandas as pd

subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

df = pd.DataFrame({
    'subject': subj_test,
    'true_label': test_metrics['labels'],
    'pred_prob': test_metrics['preds'],
})
subject_level = df.groupby('subject').agg(
    true_label=('true_label', 'first'),
    mean_pred_prob=('pred_prob', 'mean')
).reset_index()

from sklearn.metrics import roc_auc_score
subject_auroc = roc_auc_score(subject_level['true_label'], subject_level['mean_pred_prob'])
print(f"Subject-level AUROC (averaging predictions per person): {subject_auroc:.4f}")
print(f"Number of test subjects: {len(subject_level)}")

Subject-level AUROC (averaging predictions per person): 0.5893
Number of test subjects: 26


In [15]:
results = {
    'model': 'Model 2 - Insole only',
    'classification_window_level': test_cls,
    'classification_subject_level_auroc': subject_auroc,
    'regression': test_reg,
    'n_test_subjects': len(subject_level),
}
with open('/kaggle/working/model2_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

## 7. Model 3: IMU + Insole Concatenation

In [ ]:
import gc
import torch

# Delete model, optimizer, scheduler, and data-related objects
for var_name in ['model', 'optimizer', 'scheduler', 'train_loader', 'val_loader', 'test_loader',
                  'train_ds', 'val_ds', 'test_ds',
                  'train_data', 'val_data', 'test_data',
                  'X_imu_train_norm', 'X_imu_val_norm', 'X_imu_test_norm',
                  'sev_train_scaled', 'sev_val_scaled', 'sev_test_scaled',
                  'train_metrics', 'val_metrics', 'test_metrics', 'history']:
    if var_name in globals():
        del globals()[var_name]

# Force garbage collection
gc.collect()

# Clear GPU memory cache (if using CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

print("RAM/GPU cleared.")

In [2]:
DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# 2. LOAD ALL ARRAYS (memory-mapped for the big ones)
# ============================================================

X_imu_train = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")

X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")

y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")

sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")

subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

# Sanity check alignment — must match, since IMU and insole were windowed together
assert X_imu_train.shape[0] == X_insole_train.shape[0] == y_train.shape[0]
assert X_imu_val.shape[0] == X_insole_val.shape[0] == y_val.shape[0]
assert X_imu_test.shape[0] == X_insole_test.shape[0] == y_test.shape[0]
print("IMU/Insole alignment confirmed across all splits.")

print("Train:", X_imu_train.shape, X_insole_train.shape)
print("Val:  ", X_imu_val.shape, X_insole_val.shape)
print("Test: ", X_imu_test.shape, X_insole_test.shape)


# ============================================================
# 3. NORMALIZATION
#    IMU: global train-set stats (worked well in Model 1)
#    Insole: PER-WINDOW normalization (fixed Model 2's generalization issue)
# ============================================================

imu_mean = np.asarray(X_imu_train.mean(axis=(0, 1)), dtype=np.float32)
imu_std = np.asarray(X_imu_train.std(axis=(0, 1)), dtype=np.float32)
imu_std[imu_std < 1e-6] = 1.0

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std

print("IMU mean/std shape:", imu_mean.shape, imu_std.shape)
print("Severity mean:", severity_mean, "| std:", severity_std)


# ============================================================
# 4. DUAL-MODALITY DATASET
# ============================================================

class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu = X_imu
        self.X_insole = X_insole
        self.y = y
        self.severity = severity
        self.imu_mean = imu_mean
        self.imu_std = imu_std
        self.augment = augment
        self.noise_std = noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # IMU: global normalization
        x_imu = np.asarray(self.X_imu[idx], dtype=np.float32)
        x_imu = (x_imu - self.imu_mean) / self.imu_std

        # Insole: per-window normalization (fixes the generalization issue from Model 2)
        x_insole = np.asarray(self.X_insole[idx], dtype=np.float32)
        w_mean = x_insole.mean(axis=0, keepdims=True)
        w_std = x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu = torch.from_numpy(x_imu)
        x_insole = torch.from_numpy(x_insole)

        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std

        y = torch.tensor(self.y[idx], dtype=torch.float32)
        severity = torch.tensor(self.severity[idx], dtype=torch.float32)
        return x_imu, x_insole, y, severity


train_ds = DualModalityDataset(X_imu_train, X_insole_train, y_train, sev_train_scaled,
                                 imu_mean, imu_std, augment=True, noise_std=0.05)
val_ds = DualModalityDataset(X_imu_val, X_insole_val, y_val, sev_val_scaled,
                               imu_mean, imu_std, augment=False)
test_ds = DualModalityDataset(X_imu_test, X_insole_test, y_test, sev_test_scaled,
                                imu_mean, imu_std, augment=False)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

n_class0 = np.sum(y_train == 0)
n_class1 = np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)
print(f"pos_weight: {pos_weight.item():.3f}")


# ============================================================
# 5. MODEL — two separate encoders, concatenated (NO attention — that's Model 4/5)
# ============================================================

class ModalityEncoder(nn.Module):
    """Reusable single-modality encoder — same structure as Models 1 & 2's backbone."""
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)
        return h_cat


class NaiveLateFusionModel(nn.Module):
    def __init__(self, n_imu_features, n_insole_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.imu_encoder = ModalityEncoder(n_imu_features, cnn_channels, lstm_hidden, dropout)
        self.insole_encoder = ModalityEncoder(n_insole_features, cnn_channels, lstm_hidden, dropout)

        fused_dim = self.imu_encoder.output_dim + self.insole_encoder.output_dim  # simple concatenation
        self.shared = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x_imu, x_insole):
        imu_feat = self.imu_encoder(x_imu)
        insole_feat = self.insole_encoder(x_insole)
        fused = torch.cat([imu_feat, insole_feat], dim=1)  # naive concatenation — no attention
        shared_features = self.shared(fused)
        cls_logit = self.classification_head(shared_features).squeeze(-1)
        reg_output = self.regression_head(shared_features).squeeze(-1)
        return cls_logit, reg_output


n_imu_features = X_imu_train.shape[2]
n_insole_features = X_insole_train.shape[2]
model = NaiveLateFusionModel(n_imu_features, n_insole_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 6. LOSS, OPTIMIZER
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()
LAMBDA_CLS = 1.0
LAMBDA_REG = 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for x_imu_b, x_insole_b, yb, sevb in loader:
            x_imu_b, x_insole_b = x_imu_b.to(device), x_insole_b.to(device)
            yb, sevb = yb.to(device), sevb.to(device)

            cls_logit, reg_out = model(x_imu_b, x_insole_b)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = x_imu_b.size(0)
            total_loss += loss.item() * bs
            total_cls_loss += cls_loss.item() * bs
            total_reg_loss += reg_loss.item() * bs
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}


# ============================================================
# 7. TRAINING LOOP — checkpoint on val AUROC (bug fix from Model 2)
# ============================================================

N_EPOCHS = 30
PATIENCE = 10
best_val_auroc = -float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} "
          f"val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_cls['auroc'] > best_val_auroc:
        best_val_auroc = val_cls['auroc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model3_naive_fusion_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val AUROC: {best_val_auroc:.4f}")


# ============================================================
# 8. TEST EVALUATION — window-level AND subject-level
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model3_naive_fusion_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

df = pd.DataFrame({'subject': subj_test, 'true_label': test_metrics['labels'], 'pred_prob': test_metrics['preds']})
subject_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
subject_auroc = roc_auc_score(subject_level['true_label'], subject_level['mean_pred_prob'])

print("MODEL 3 (NAIVE LATE FUSION) — TEST RESULTS")
print(f"Window-level  — Accuracy: {test_cls['accuracy']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"Subject-level — AUROC: {subject_auroc:.4f} (n={len(subject_level)})")
print(f"Regression    — MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {
    'model': 'Model 3 - Naive late fusion',
    'classification_window_level': test_cls,
    'classification_subject_level_auroc': subject_auroc,
    'regression': test_reg,
    'n_test_subjects': len(subject_level),
}
with open('/kaggle/working/model3_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

Using device: cuda
IMU/Insole alignment confirmed across all splits.
Train: (17066, 300, 286) (17066, 300, 50)
Val:   (4006, 300, 286) (4006, 300, 50)
Test:  (3493, 300, 286) (3493, 300, 50)
IMU mean/std shape: (286,) (286,)
Severity mean: 18.17444 | std: 13.93664
pos_weight: 0.757
NaiveLateFusionModel(
  (imu_encoder): ModalityEncoder(
    (conv1): Conv1d(286, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout_conv): Dropout(p=0.25, inplace=False)
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  )
  (insole_encoder): ModalityEncoder(
    (conv1): Conv1d(50, 32, kernel_size=(5,), stride=(1,), padding=(2

In [1]:
DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

# **Model 4**

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1-4. DATA LOADING, NORMALIZATION, DATASET — identical to Model 3
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


X_imu_train = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")
X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")
subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

assert X_imu_train.shape[0] == X_insole_train.shape[0] == y_train.shape[0]
print("Alignment confirmed. Train:", X_imu_train.shape, X_insole_train.shape)

imu_mean = np.asarray(X_imu_train.mean(axis=(0, 1)), dtype=np.float32)
imu_std = np.asarray(X_imu_train.std(axis=(0, 1)), dtype=np.float32)
imu_std[imu_std < 1e-6] = 1.0

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std


class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu, self.X_insole = X_imu, X_insole
        self.y, self.severity = y, severity
        self.imu_mean, self.imu_std = imu_mean, imu_std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_imu = np.asarray(self.X_imu[idx], dtype=np.float32)
        x_imu = (x_imu - self.imu_mean) / self.imu_std

        x_insole = np.asarray(self.X_insole[idx], dtype=np.float32)
        w_mean = x_insole.mean(axis=0, keepdims=True)
        w_std = x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu, x_insole = torch.from_numpy(x_imu), torch.from_numpy(x_insole)
        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std

        y = torch.tensor(self.y[idx], dtype=torch.float32)
        severity = torch.tensor(self.severity[idx], dtype=torch.float32)
        return x_imu, x_insole, y, severity


train_ds = DualModalityDataset(X_imu_train, X_insole_train, y_train, sev_train_scaled, imu_mean, imu_std, augment=True)
val_ds = DualModalityDataset(X_imu_val, X_insole_val, y_val, sev_val_scaled, imu_mean, imu_std, augment=False)
test_ds = DualModalityDataset(X_imu_test, X_insole_test, y_test, sev_test_scaled, imu_mean, imu_std, augment=False)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

n_class0, n_class1 = np.sum(y_train == 0), np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)
print(f"pos_weight: {pos_weight.item():.3f}")


# ============================================================
# 5. GaitFuseNet — MODEL 4: Cross-modal attention fusion
# ============================================================

class ModalityEncoderSeq(nn.Module):
    """Same CNN+BiLSTM backbone as before, but now returns the FULL sequence
    (not just final hidden state), so cross-attention can operate across time."""
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)  # (batch, T', channels)
        seq_out, _ = self.bilstm(x)  # (batch, T', lstm_hidden*2) — FULL sequence, not just final state
        return seq_out


class CrossModalAttentionFusion(nn.Module):
    """The core novel component: IMU and insole sequences attend to EACH OTHER
    bidirectionally, rather than being processed independently then concatenated."""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.imu_attends_insole = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.insole_attends_imu = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_imu = nn.LayerNorm(d_model)
        self.norm_insole = nn.LayerNorm(d_model)

    def forward(self, imu_seq, insole_seq):
        # IMU queries, insole provides keys/values — "what was insole doing when IMU moved like this?"
        imu_attended, imu_attn_weights = self.imu_attends_insole(imu_seq, insole_seq, insole_seq)
        imu_out = self.norm_imu(imu_seq + imu_attended)  # residual connection + layer norm

        # Insole queries, IMU provides keys/values — the reverse direction
        insole_attended, insole_attn_weights = self.insole_attends_imu(insole_seq, imu_seq, imu_seq)
        insole_out = self.norm_insole(insole_seq + insole_attended)

        return imu_out, insole_out, imu_attn_weights, insole_attn_weights


class GaitFuseNetModel4(nn.Module):
    def __init__(self, n_imu_features, n_insole_features, cnn_channels=32, lstm_hidden=64,
                 n_heads=4, dropout=0.5):
        super().__init__()
        self.imu_encoder = ModalityEncoderSeq(n_imu_features, cnn_channels, lstm_hidden, dropout)
        self.insole_encoder = ModalityEncoderSeq(n_insole_features, cnn_channels, lstm_hidden, dropout)

        d_model = lstm_hidden * 2  # both encoders output the same dimension, required for attention
        self.cross_attn = CrossModalAttentionFusion(d_model, n_heads=n_heads, dropout=0.1)

        self.shared = nn.Sequential(
            nn.Linear(d_model * 2, 64),  # concat pooled IMU + insole attended features
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x_imu, x_insole, return_attention=False):
        imu_seq = self.imu_encoder(x_imu)        # (batch, T', d_model)
        insole_seq = self.insole_encoder(x_insole)

        imu_out, insole_out, imu_attn_w, insole_attn_w = self.cross_attn(imu_seq, insole_seq)

        # Pool over time to get one fixed-size vector per modality
        imu_pooled = imu_out.mean(dim=1)
        insole_pooled = insole_out.mean(dim=1)

        fused = torch.cat([imu_pooled, insole_pooled], dim=1)
        shared_features = self.shared(fused)
        cls_logit = self.classification_head(shared_features).squeeze(-1)
        reg_output = self.regression_head(shared_features).squeeze(-1)

        if return_attention:
            return cls_logit, reg_output, imu_attn_w, insole_attn_w
        return cls_logit, reg_output


n_imu_features = X_imu_train.shape[2]
n_insole_features = X_insole_train.shape[2]
model = GaitFuseNetModel4(n_imu_features, n_insole_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 6. LOSS, OPTIMIZER, TRAINING LOOP — identical structure to Model 3
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()
LAMBDA_CLS, LAMBDA_REG = 1.0, 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for x_imu_b, x_insole_b, yb, sevb in loader:
            x_imu_b, x_insole_b = x_imu_b.to(device), x_insole_b.to(device)
            yb, sevb = yb.to(device), sevb.to(device)

            cls_logit, reg_out = model(x_imu_b, x_insole_b)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = x_imu_b.size(0)
            total_loss += loss.item() * bs
            total_cls_loss += cls_loss.item() * bs
            total_reg_loss += reg_loss.item() * bs
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}

N_EPOCHS = 30
PATIENCE = 10
best_val_auroc = -float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} "
          f"val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_cls['auroc'] > best_val_auroc:
        best_val_auroc = val_cls['auroc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model4_gaitfusenet_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val AUROC: {best_val_auroc:.4f}")


# ============================================================
# 7. TEST EVALUATION
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model4_gaitfusenet_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

df = pd.DataFrame({'subject': subj_test, 'true_label': test_metrics['labels'], 'pred_prob': test_metrics['preds']})
subject_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
subject_auroc = roc_auc_score(subject_level['true_label'], subject_level['mean_pred_prob'])

print("MODEL 4 (GAITFUSENET — CROSS-MODAL ATTENTION) — TEST RESULTS")
print(f"Window-level  — Accuracy: {test_cls['accuracy']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"Subject-level — AUROC: {subject_auroc:.4f} (n={len(subject_level)})")
print(f"Regression    — MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {
    'model': 'Model 4 - GaitFuseNet (cross-modal attention)',
    'classification_window_level': test_cls,
    'classification_subject_level_auroc': subject_auroc,
    'regression': test_reg,
    'n_test_subjects': len(subject_level),
}
with open('/kaggle/working/model4_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

Using device: cuda
Alignment confirmed. Train: (17066, 300, 286) (17066, 300, 50)
pos_weight: 0.757
GaitFuseNetModel4(
  (imu_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(286, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout_conv): Dropout(p=0.25, inplace=False)
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  )
  (insole_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(50, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): B

# **Model 5**

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1-4. DATA LOADING, NORMALIZATION, DATASET — identical to Model 3
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


X_imu_train = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")
X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")
subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

assert X_imu_train.shape[0] == X_insole_train.shape[0] == y_train.shape[0]
print("Alignment confirmed. Train:", X_imu_train.shape, X_insole_train.shape)

imu_mean = np.asarray(X_imu_train.mean(axis=(0, 1)), dtype=np.float32)
imu_std = np.asarray(X_imu_train.std(axis=(0, 1)), dtype=np.float32)
imu_std[imu_std < 1e-6] = 1.0

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std


class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu, self.X_insole = X_imu, X_insole
        self.y, self.severity = y, severity
        self.imu_mean, self.imu_std = imu_mean, imu_std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_imu = np.asarray(self.X_imu[idx], dtype=np.float32)
        x_imu = (x_imu - self.imu_mean) / self.imu_std

        x_insole = np.asarray(self.X_insole[idx], dtype=np.float32)
        w_mean = x_insole.mean(axis=0, keepdims=True)
        w_std = x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu, x_insole = torch.from_numpy(x_imu), torch.from_numpy(x_insole)
        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std

        y = torch.tensor(self.y[idx], dtype=torch.float32)
        severity = torch.tensor(self.severity[idx], dtype=torch.float32)
        return x_imu, x_insole, y, severity


train_ds = DualModalityDataset(X_imu_train, X_insole_train, y_train, sev_train_scaled, imu_mean, imu_std, augment=True)
val_ds = DualModalityDataset(X_imu_val, X_insole_val, y_val, sev_val_scaled, imu_mean, imu_std, augment=False)
test_ds = DualModalityDataset(X_imu_test, X_insole_test, y_test, sev_test_scaled, imu_mean, imu_std, augment=False)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

n_class0, n_class1 = np.sum(y_train == 0), np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)
print(f"pos_weight: {pos_weight.item():.3f}")


# ============================================================
# 5. GaitFuseNet — MODEL 4: Cross-modal attention fusion
# ============================================================

class ModalityEncoderSeq(nn.Module):
    """Same CNN+BiLSTM backbone as before, but now returns the FULL sequence
    (not just final hidden state), so cross-attention can operate across time."""
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)  # (batch, T', channels)
        seq_out, _ = self.bilstm(x)  # (batch, T', lstm_hidden*2) — FULL sequence, not just final state
        return seq_out


class CrossModalAttentionFusion(nn.Module):
    """The core novel component: IMU and insole sequences attend to EACH OTHER
    bidirectionally, rather than being processed independently then concatenated."""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.imu_attends_insole = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.insole_attends_imu = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_imu = nn.LayerNorm(d_model)
        self.norm_insole = nn.LayerNorm(d_model)

    def forward(self, imu_seq, insole_seq):
        # IMU queries, insole provides keys/values — "what was insole doing when IMU moved like this?"
        imu_attended, imu_attn_weights = self.imu_attends_insole(imu_seq, insole_seq, insole_seq)
        imu_out = self.norm_imu(imu_seq + imu_attended)  # residual connection + layer norm

        # Insole queries, IMU provides keys/values — the reverse direction
        insole_attended, insole_attn_weights = self.insole_attends_imu(insole_seq, imu_seq, imu_seq)
        insole_out = self.norm_insole(insole_seq + insole_attended)

        return imu_out, insole_out, imu_attn_weights, insole_attn_weights


class GaitFuseNetModel4(nn.Module):
    def __init__(self, n_imu_features, n_insole_features, cnn_channels=32, lstm_hidden=64,
                 n_heads=4, dropout=0.5):
        super().__init__()
        self.imu_encoder = ModalityEncoderSeq(n_imu_features, cnn_channels, lstm_hidden, dropout)
        self.insole_encoder = ModalityEncoderSeq(n_insole_features, cnn_channels, lstm_hidden, dropout)

        d_model = lstm_hidden * 2  # both encoders output the same dimension, required for attention
        self.cross_attn = CrossModalAttentionFusion(d_model, n_heads=n_heads, dropout=0.1)

        self.fusion_dropout = nn.Dropout(0.30)
        self.shared = nn.Sequential(
                nn.Linear(d_model * 2, 64),
                nn.ReLU(),
                nn.Dropout(0.50),
                )
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x_imu, x_insole, return_attention=False):
        imu_seq = self.imu_encoder(x_imu)        # (batch, T', d_model)
        insole_seq = self.insole_encoder(x_insole)

        imu_out, insole_out, imu_attn_w, insole_attn_w = self.cross_attn(imu_seq, insole_seq)

        # Pool over time to get one fixed-size vector per modality
        imu_pooled = imu_out.mean(dim=1)
        insole_pooled = insole_out.mean(dim=1)

        fused = torch.cat([imu_pooled, insole_pooled], dim=1)
        fused = self.fusion_dropout(fused)
        shared_features = self.shared(fused)
        cls_logit = self.classification_head(shared_features).squeeze(-1)
        reg_output = self.regression_head(shared_features).squeeze(-1)

        if return_attention:
            return cls_logit, reg_output, imu_attn_w, insole_attn_w
        return cls_logit, reg_output


n_imu_features = X_imu_train.shape[2]
n_insole_features = X_insole_train.shape[2]
model = GaitFuseNetModel4(n_imu_features, n_insole_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 6. LOSS, OPTIMIZER, TRAINING LOOP — identical structure to Model 3
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()
LAMBDA_CLS, LAMBDA_REG = 1.0, 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for x_imu_b, x_insole_b, yb, sevb in loader:
            x_imu_b, x_insole_b = x_imu_b.to(device), x_insole_b.to(device)
            yb, sevb = yb.to(device), sevb.to(device)

            cls_logit, reg_out = model(x_imu_b, x_insole_b)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )
                
                optimizer.step()

            bs = x_imu_b.size(0)
            total_loss += loss.item() * bs
            total_cls_loss += cls_loss.item() * bs
            total_reg_loss += reg_loss.item() * bs
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}

N_EPOCHS = 30
PATIENCE = 10
best_val_auroc = -float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} "
          f"val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_cls['auroc'] > best_val_auroc:
        best_val_auroc = val_cls['auroc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model5_gaitfusenet_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val AUROC: {best_val_auroc:.4f}")


# ============================================================
# 7. TEST EVALUATION
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model5_gaitfusenet_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

df = pd.DataFrame({'subject': subj_test, 'true_label': test_metrics['labels'], 'pred_prob': test_metrics['preds']})
subject_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
subject_auroc = roc_auc_score(subject_level['true_label'], subject_level['mean_pred_prob'])

print("MODEL 4 (GAITFUSENET — CROSS-MODAL ATTENTION) — TEST RESULTS")
print(f"Window-level  — Accuracy: {test_cls['accuracy']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"Subject-level — AUROC: {subject_auroc:.4f} (n={len(subject_level)})")
print(f"Regression    — MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {
    'model': 'Model 5 - GaitFuseNet (cross-modal attention)',
    'classification_window_level': test_cls,
    'classification_subject_level_auroc': subject_auroc,
    'regression': test_reg,
    'n_test_subjects': len(subject_level),
}
with open('/kaggle/working/model5_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

Using device: cuda
Alignment confirmed. Train: (17066, 300, 286) (17066, 300, 50)
pos_weight: 0.757
GaitFuseNetModel4(
  (imu_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(286, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout_conv): Dropout(p=0.25, inplace=False)
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  )
  (insole_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(50, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): B

# **Model 6**

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1-4. DATA LOADING, NORMALIZATION, DATASET — identical to Model 3
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


X_imu_train = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")
X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")
subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

assert X_imu_train.shape[0] == X_insole_train.shape[0] == y_train.shape[0]
print("Alignment confirmed. Train:", X_imu_train.shape, X_insole_train.shape)

imu_mean = np.asarray(X_imu_train.mean(axis=(0, 1)), dtype=np.float32)
imu_std = np.asarray(X_imu_train.std(axis=(0, 1)), dtype=np.float32)
imu_std[imu_std < 1e-6] = 1.0

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std


class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu, self.X_insole = X_imu, X_insole
        self.y, self.severity = y, severity
        self.imu_mean, self.imu_std = imu_mean, imu_std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_imu = np.asarray(self.X_imu[idx], dtype=np.float32)
        x_imu = (x_imu - self.imu_mean) / self.imu_std

        x_insole = np.asarray(self.X_insole[idx], dtype=np.float32)
        w_mean = x_insole.mean(axis=0, keepdims=True)
        w_std = x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu, x_insole = torch.from_numpy(x_imu), torch.from_numpy(x_insole)
        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std

        y = torch.tensor(self.y[idx], dtype=torch.float32)
        severity = torch.tensor(self.severity[idx], dtype=torch.float32)
        return x_imu, x_insole, y, severity


train_ds = DualModalityDataset(X_imu_train, X_insole_train, y_train, sev_train_scaled, imu_mean, imu_std, augment=True)
val_ds = DualModalityDataset(X_imu_val, X_insole_val, y_val, sev_val_scaled, imu_mean, imu_std, augment=False)
test_ds = DualModalityDataset(X_imu_test, X_insole_test, y_test, sev_test_scaled, imu_mean, imu_std, augment=False)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

n_class0, n_class1 = np.sum(y_train == 0), np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)
print(f"pos_weight: {pos_weight.item():.3f}")


# ============================================================
# 5. GaitFuseNet — MODEL 4: Cross-modal attention fusion
# ============================================================

class ModalityEncoderSeq(nn.Module):
    """Same CNN+BiLSTM backbone as before, but now returns the FULL sequence
    (not just final hidden state), so cross-attention can operate across time."""
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)  # (batch, T', channels)
        seq_out, _ = self.bilstm(x)  # (batch, T', lstm_hidden*2) — FULL sequence, not just final state
        return seq_out


class CrossModalAttentionFusion(nn.Module):
    """The core novel component: IMU and insole sequences attend to EACH OTHER
    bidirectionally, rather than being processed independently then concatenated."""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.imu_attends_insole = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.insole_attends_imu = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_imu = nn.LayerNorm(d_model)
        self.norm_insole = nn.LayerNorm(d_model)

    def forward(self, imu_seq, insole_seq):
        # IMU queries, insole provides keys/values — "what was insole doing when IMU moved like this?"
        imu_attended, imu_attn_weights = self.imu_attends_insole(imu_seq, insole_seq, insole_seq)
        imu_out = self.norm_imu(imu_seq + imu_attended)  # residual connection + layer norm

        # Insole queries, IMU provides keys/values — the reverse direction
        insole_attended, insole_attn_weights = self.insole_attends_imu(insole_seq, imu_seq, imu_seq)
        insole_out = self.norm_insole(insole_seq + insole_attended)

        return imu_out, insole_out, imu_attn_weights, insole_attn_weights


class GaitFuseNetModel4(nn.Module):
    def __init__(self, n_imu_features, n_insole_features, cnn_channels=24, lstm_hidden=48,
                 n_heads=4, dropout=0.5):
        super().__init__()
        self.imu_encoder = ModalityEncoderSeq(n_imu_features, cnn_channels, lstm_hidden, dropout)
        self.insole_encoder = ModalityEncoderSeq(n_insole_features, cnn_channels, lstm_hidden, dropout)

        d_model = lstm_hidden * 2  # both encoders output the same dimension, required for attention
        self.cross_attn = CrossModalAttentionFusion(d_model, n_heads=n_heads, dropout=0.1)

        self.shared = nn.Sequential(
            nn.Linear(d_model * 2, 64),  # concat pooled IMU + insole attended features
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x_imu, x_insole, return_attention=False):
        imu_seq = self.imu_encoder(x_imu)        # (batch, T', d_model)
        insole_seq = self.insole_encoder(x_insole)

        imu_out, insole_out, imu_attn_w, insole_attn_w = self.cross_attn(imu_seq, insole_seq)

        # Pool over time to get one fixed-size vector per modality
        imu_pooled = imu_out.mean(dim=1)
        insole_pooled = insole_out.mean(dim=1)

        fused = torch.cat([imu_pooled, insole_pooled], dim=1)
        shared_features = self.shared(fused)
        cls_logit = self.classification_head(shared_features).squeeze(-1)
        reg_output = self.regression_head(shared_features).squeeze(-1)

        if return_attention:
            return cls_logit, reg_output, imu_attn_w, insole_attn_w
        return cls_logit, reg_output


n_imu_features = X_imu_train.shape[2]
n_insole_features = X_insole_train.shape[2]
model = GaitFuseNetModel4(n_imu_features, n_insole_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 6. LOSS, OPTIMIZER, TRAINING LOOP — identical structure to Model 3
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()
LAMBDA_CLS, LAMBDA_REG = 1.0, 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for x_imu_b, x_insole_b, yb, sevb in loader:
            x_imu_b, x_insole_b = x_imu_b.to(device), x_insole_b.to(device)
            yb, sevb = yb.to(device), sevb.to(device)

            cls_logit, reg_out = model(x_imu_b, x_insole_b)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = x_imu_b.size(0)
            total_loss += loss.item() * bs
            total_cls_loss += cls_loss.item() * bs
            total_reg_loss += reg_loss.item() * bs
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}

N_EPOCHS = 30
PATIENCE = 10
best_val_auroc = -float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} "
          f"val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_cls['auroc'] > best_val_auroc:
        best_val_auroc = val_cls['auroc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model6_gaitfusenet_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val AUROC: {best_val_auroc:.4f}")


# ============================================================
# 7. TEST EVALUATION
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model6_gaitfusenet_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

df = pd.DataFrame({'subject': subj_test, 'true_label': test_metrics['labels'], 'pred_prob': test_metrics['preds']})
subject_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
subject_auroc = roc_auc_score(subject_level['true_label'], subject_level['mean_pred_prob'])

print("MODEL 6 (GAITFUSENET — CROSS-MODAL ATTENTION) — TEST RESULTS")
print(f"Window-level  — Accuracy: {test_cls['accuracy']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"Subject-level — AUROC: {subject_auroc:.4f} (n={len(subject_level)})")
print(f"Regression    — MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {
    'model': 'Model 6 - GaitFuseNet (cross-modal attention)',
    'classification_window_level': test_cls,
    'classification_subject_level_auroc': subject_auroc,
    'regression': test_reg,
    'n_test_subjects': len(subject_level),
}
with open('/kaggle/working/model6_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

Using device: cuda
Alignment confirmed. Train: (17066, 300, 286) (17066, 300, 50)
pos_weight: 0.757
GaitFuseNetModel4(
  (imu_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(286, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(24, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout_conv): Dropout(p=0.25, inplace=False)
    (bilstm): LSTM(24, 48, batch_first=True, bidirectional=True)
  )
  (insole_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(50, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(24, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): B

# **Model 7**

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1-4. DATA LOADING, NORMALIZATION, DATASET — identical to Model 3
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


X_imu_train = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")
X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")
subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

assert X_imu_train.shape[0] == X_insole_train.shape[0] == y_train.shape[0]
print("Alignment confirmed. Train:", X_imu_train.shape, X_insole_train.shape)

imu_mean = np.asarray(X_imu_train.mean(axis=(0, 1)), dtype=np.float32)
imu_std = np.asarray(X_imu_train.std(axis=(0, 1)), dtype=np.float32)
imu_std[imu_std < 1e-6] = 1.0

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std


class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu, self.X_insole = X_imu, X_insole
        self.y, self.severity = y, severity
        self.imu_mean, self.imu_std = imu_mean, imu_std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_imu = np.asarray(self.X_imu[idx], dtype=np.float32)
        x_imu = (x_imu - self.imu_mean) / self.imu_std

        x_insole = np.asarray(self.X_insole[idx], dtype=np.float32)
        w_mean = x_insole.mean(axis=0, keepdims=True)
        w_std = x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu, x_insole = torch.from_numpy(x_imu), torch.from_numpy(x_insole)
        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std

        y = torch.tensor(self.y[idx], dtype=torch.float32)
        severity = torch.tensor(self.severity[idx], dtype=torch.float32)
        return x_imu, x_insole, y, severity


train_ds = DualModalityDataset(X_imu_train, X_insole_train, y_train, sev_train_scaled, imu_mean, imu_std, augment=True)
val_ds = DualModalityDataset(X_imu_val, X_insole_val, y_val, sev_val_scaled, imu_mean, imu_std, augment=False)
test_ds = DualModalityDataset(X_imu_test, X_insole_test, y_test, sev_test_scaled, imu_mean, imu_std, augment=False)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

n_class0, n_class1 = np.sum(y_train == 0), np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)
print(f"pos_weight: {pos_weight.item():.3f}")


# ============================================================
# 5. GaitFuseNet — MODEL 4: Cross-modal attention fusion
# ============================================================

class ModalityEncoderSeq(nn.Module):
    """Same CNN+BiLSTM backbone as before, but now returns the FULL sequence
    (not just final hidden state), so cross-attention can operate across time."""
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)  # (batch, T', channels)
        seq_out, _ = self.bilstm(x)  # (batch, T', lstm_hidden*2) — FULL sequence, not just final state
        return seq_out


class CrossModalAttentionFusion(nn.Module):
    """The core novel component: IMU and insole sequences attend to EACH OTHER
    bidirectionally, rather than being processed independently then concatenated."""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.imu_attends_insole = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.insole_attends_imu = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_imu = nn.LayerNorm(d_model)
        self.norm_insole = nn.LayerNorm(d_model)

    def forward(self, imu_seq, insole_seq):
        # IMU queries, insole provides keys/values — "what was insole doing when IMU moved like this?"
        imu_attended, imu_attn_weights = self.imu_attends_insole(imu_seq, insole_seq, insole_seq)
        imu_out = self.norm_imu(imu_seq + imu_attended)  # residual connection + layer norm

        # Insole queries, IMU provides keys/values — the reverse direction
        insole_attended, insole_attn_weights = self.insole_attends_imu(insole_seq, imu_seq, imu_seq)
        insole_out = self.norm_insole(insole_seq + insole_attended)

        return imu_out, insole_out, imu_attn_weights, insole_attn_weights
        
class GatedFusion(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

    def forward(self, imu_feat, insole_feat):
        # imu_feat:    [B, d_model]
        # insole_feat: [B, d_model]

        combined = torch.cat([imu_feat, insole_feat], dim=1)

        # Values between 0 and 1
        gate = self.gate(combined)

        # gate close to 1 -> rely more on IMU
        # gate close to 0 -> rely more on insole
        fused = gate * imu_feat + (1.0 - gate) * insole_feat

        return fused, gate


class GaitFuseNetGated(nn.Module):
    def __init__(
        self,
        imu_features=286,
        insole_features=50,
        cnn_channels=32,
        lstm_hidden=64,
        n_heads=4,
        dropout=0.5
    ):
        super().__init__()

        # Separate modality encoders
        self.imu_encoder = ModalityEncoderSeq(
            imu_features,
            cnn_channels,
            lstm_hidden,
            dropout=0.25
        )

        self.insole_encoder = ModalityEncoderSeq(
            insole_features,
            cnn_channels,
            lstm_hidden,
            dropout=0.25
        )

        d_model = lstm_hidden * 2

        # Existing bidirectional cross-modal attention
        self.cross_attn = CrossModalAttentionFusion(
            d_model=d_model,
            n_heads=n_heads,
            dropout=0.1
        )

        # NEW: gated fusion
        self.gated_fusion = GatedFusion(d_model)

        # IMPORTANT:
        # Gated fusion outputs d_model, not 2*d_model
        self.shared = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, imu, insole):

        # 1. Encode both modalities
        imu_seq = self.imu_encoder(imu)
        insole_seq = self.insole_encoder(insole)

        # 2. Cross-modal attention
        imu_out, insole_out, imu_attn, insole_attn = \
            self.cross_attn(imu_seq, insole_seq)

        # 3. Temporal mean pooling
        imu_feat = imu_out.mean(dim=1)
        insole_feat = insole_out.mean(dim=1)

        # 4. GATED FUSION
        fused, gate = self.gated_fusion(
            imu_feat,
            insole_feat
        )

        # 5. Shared representation
        shared_features = self.shared(fused)

        # 6. Prediction heads
        cls_logit = self.classification_head(
            shared_features
        ).squeeze(-1)

        reg_output = self.regression_head(
            shared_features
        ).squeeze(-1)

        return cls_logit, reg_output, gate
    

n_imu_features = X_imu_train.shape[2]
n_insole_features = X_insole_train.shape[2]

model = GaitFuseNetGated(
    imu_features=286,
    insole_features=50,
    cnn_channels=32,
    lstm_hidden=64,
    n_heads=4,
    dropout=0.5
).to(device)
model = GaitFuseNetModel4(n_imu_features, n_insole_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 6. LOSS, OPTIMIZER, TRAINING LOOP — identical structure to Model 3
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()
LAMBDA_CLS, LAMBDA_REG = 1.0, 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for x_imu_b, x_insole_b, yb, sevb in loader:
            x_imu_b, x_insole_b = x_imu_b.to(device), x_insole_b.to(device)
            yb, sevb = yb.to(device), sevb.to(device)

            cls_logit, reg_out = model(x_imu_b, x_insole_b)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = x_imu_b.size(0)
            total_loss += loss.item() * bs
            total_cls_loss += cls_loss.item() * bs
            total_reg_loss += reg_loss.item() * bs
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}

N_EPOCHS = 30
PATIENCE = 10
best_val_auroc = -float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} "
          f"val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_cls['auroc'] > best_val_auroc:
        best_val_auroc = val_cls['auroc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model7_gaitfusenet_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val AUROC: {best_val_auroc:.4f}")


# ============================================================
# 7. TEST EVALUATION
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model7_gaitfusenet_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

df = pd.DataFrame({'subject': subj_test, 'true_label': test_metrics['labels'], 'pred_prob': test_metrics['preds']})
subject_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
subject_auroc = roc_auc_score(subject_level['true_label'], subject_level['mean_pred_prob'])

print("MODEL 7 (GAITFUSENET — CROSS-MODAL ATTENTION) — TEST RESULTS")
print(f"Window-level  — Accuracy: {test_cls['accuracy']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"Subject-level — AUROC: {subject_auroc:.4f} (n={len(subject_level)})")
print(f"Regression    — MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {
    'model': 'Model 7 - GaitFuseNet (cross-modal attention)',
    'classification_window_level': test_cls,
    'classification_subject_level_auroc': subject_auroc,
    'regression': test_reg,
    'n_test_subjects': len(subject_level),
}
with open('/kaggle/working/model7_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

Using device: cuda
Alignment confirmed. Train: (17066, 300, 286) (17066, 300, 50)
pos_weight: 0.757
GaitFuseNetModel4(
  (imu_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(286, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(24, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout_conv): Dropout(p=0.25, inplace=False)
    (bilstm): LSTM(24, 48, batch_first=True, bidirectional=True)
  )
  (insole_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(50, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(24, 24, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): B

In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 1. CONFIRM SENSOR ORDER, THEN SLICE TO 3 SENSORS ONLY
# ============================================================

IMU_SENSOR_NAMES = [
    'LowerBack', 'R_Wrist', 'L_Wrist', 'R_MidLatThigh', 'L_MidLatThigh',
    'R_LatShank', 'L_LatShank', 'R_DorsalFoot', 'L_DorsalFoot',
    'R_Ankle', 'L_Ankle', 'Xiphoid', 'Forehead'
]
# LowerBack=0, R_Ankle=9, L_Ankle=10 — confirm this matches your imu_cols order before trusting the slice
keep_sensor_indices = [0, 9, 10]
keep_col_indices = np.concatenate([np.arange(i * 22, (i + 1) * 22) for i in keep_sensor_indices])
print(f"Keeping {len(keep_col_indices)} IMU columns (3 sensors x 22 each): {[IMU_SENSOR_NAMES[i] for i in keep_sensor_indices]}")

DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"
X_imu_train_full = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val_full = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test_full = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")

X_imu_train_small = np.asarray(X_imu_train_full[:, :, keep_col_indices], dtype=np.float32)
X_imu_val_small = np.asarray(X_imu_val_full[:, :, keep_col_indices], dtype=np.float32)
X_imu_test_small = np.asarray(X_imu_test_full[:, :, keep_col_indices], dtype=np.float32)

print("Reduced IMU shapes:", X_imu_train_small.shape, X_imu_val_small.shape, X_imu_test_small.shape)

X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")

y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")
subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

imu_mean_small = X_imu_train_small.mean(axis=(0, 1), keepdims=False).astype(np.float32)
imu_std_small = X_imu_train_small.std(axis=(0, 1), keepdims=False).astype(np.float32)
imu_std_small[imu_std_small < 1e-6] = 1.0

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std

n_class0, n_class1 = np.sum(y_train == 0), np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)


# ============================================================
# 2. DATASETS (single-modality and dual-modality versions)
# ============================================================

class SingleModalityDataset(Dataset):
    def __init__(self, X, y, severity, mean, std, augment=False, noise_std=0.05):
        self.X, self.y, self.severity = X, y, severity
        self.mean, self.std = mean, std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = (self.X[idx] - self.mean) / self.std
        x = torch.from_numpy(x.astype(np.float32))
        if self.augment:
            x = x + torch.randn_like(x) * self.noise_std
        return x, torch.tensor(self.y[idx], dtype=torch.float32), torch.tensor(self.severity[idx], dtype=torch.float32)


class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu, self.X_insole = X_imu, X_insole
        self.y, self.severity = y, severity
        self.imu_mean, self.imu_std = imu_mean, imu_std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_imu = (self.X_imu[idx] - self.imu_mean) / self.imu_std
        x_insole = np.asarray(self.X_insole[idx], dtype=np.float32)
        w_mean = x_insole.mean(axis=0, keepdims=True)
        w_std = x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu = torch.from_numpy(x_imu.astype(np.float32))
        x_insole = torch.from_numpy(x_insole)
        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std
        return x_imu, x_insole, torch.tensor(self.y[idx], dtype=torch.float32), torch.tensor(self.severity[idx], dtype=torch.float32)


# ============================================================
# 3. MODEL DEFINITIONS (reduced-IMU-only, naive fusion, cross-attention fusion)
# ============================================================

class SingleModalityModel(nn.Module):
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.shared = nn.Sequential(nn.Linear(lstm_hidden * 2, 64), nn.ReLU(), nn.Dropout(dropout))
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)
        shared = self.shared(h_cat)
        return self.classification_head(shared).squeeze(-1), self.regression_head(shared).squeeze(-1)


class ModalityEncoderSeq(nn.Module):
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        seq_out, _ = self.bilstm(x)
        return seq_out


class NaiveLateFusionModel(nn.Module):
    def __init__(self, n_imu, n_insole, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.imu_encoder = ModalityEncoderSeq(n_imu, cnn_channels, lstm_hidden, dropout)
        self.insole_encoder = ModalityEncoderSeq(n_insole, cnn_channels, lstm_hidden, dropout)
        fused_dim = self.imu_encoder.output_dim + self.insole_encoder.output_dim
        self.shared = nn.Sequential(nn.Linear(fused_dim, 64), nn.ReLU(), nn.Dropout(dropout))
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x_imu, x_insole):
        imu_feat = self.imu_encoder(x_imu).mean(dim=1)
        insole_feat = self.insole_encoder(x_insole).mean(dim=1)
        fused = torch.cat([imu_feat, insole_feat], dim=1)
        shared = self.shared(fused)
        return self.classification_head(shared).squeeze(-1), self.regression_head(shared).squeeze(-1)


class CrossModalAttentionFusion(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.imu_attends_insole = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.insole_attends_imu = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_imu = nn.LayerNorm(d_model)
        self.norm_insole = nn.LayerNorm(d_model)

    def forward(self, imu_seq, insole_seq):
        imu_attended, _ = self.imu_attends_insole(imu_seq, insole_seq, insole_seq)
        imu_out = self.norm_imu(imu_seq + imu_attended)
        insole_attended, _ = self.insole_attends_imu(insole_seq, imu_seq, imu_seq)
        insole_out = self.norm_insole(insole_seq + insole_attended)
        return imu_out, insole_out


class GaitFuseNetModel(nn.Module):
    def __init__(self, n_imu, n_insole, cnn_channels=32, lstm_hidden=64, n_heads=4, dropout=0.5):
        super().__init__()
        self.imu_encoder = ModalityEncoderSeq(n_imu, cnn_channels, lstm_hidden, dropout)
        self.insole_encoder = ModalityEncoderSeq(n_insole, cnn_channels, lstm_hidden, dropout)
        d_model = lstm_hidden * 2
        self.cross_attn = CrossModalAttentionFusion(d_model, n_heads=n_heads, dropout=0.1)
        self.shared = nn.Sequential(nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Dropout(dropout))
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x_imu, x_insole):
        imu_seq = self.imu_encoder(x_imu)
        insole_seq = self.insole_encoder(x_insole)
        imu_out, insole_out = self.cross_attn(imu_seq, insole_seq)
        fused = torch.cat([imu_out.mean(dim=1), insole_out.mean(dim=1)], dim=1)
        shared = self.shared(fused)
        return self.classification_head(shared).squeeze(-1), self.regression_head(shared).squeeze(-1)


# ============================================================
# 4. GENERIC TRAIN/EVAL LOOP (works for single- or dual-input models)
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()

def unscale_severity(s, mean, std):
    return s * std + mean

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_s, pred_s, mean, std):
    t, p = unscale_severity(true_s, mean, std), unscale_severity(pred_s, mean, std)
    return {'mae': np.mean(np.abs(t - p)), 'rmse': np.sqrt(np.mean((t - p) ** 2))}

def run_epoch_generic(model, loader, dual_input, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []
    with torch.set_grad_enabled(train):
        for batch in loader:
            if dual_input:
                x1, x2, yb, sevb = batch
                x1, x2, yb, sevb = x1.to(device), x2.to(device), yb.to(device), sevb.to(device)
                cls_logit, reg_out = model(x1, x2)
                bs = x1.size(0)
            else:
                x1, yb, sevb = batch
                x1, yb, sevb = x1.to(device), yb.to(device), sevb.to(device)
                cls_logit, reg_out = model(x1)
                bs = x1.size(0)

            loss = cls_criterion(cls_logit, yb) + reg_criterion(reg_out, sevb)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()

            total_loss += loss.item() * bs
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())
    n = len(loader.dataset)
    return {'loss': total_loss / n, 'preds': np.array(all_preds), 'labels': np.array(all_labels),
            'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true)}

def train_and_evaluate(model, model_name, train_loader, val_loader, test_loader, dual_input, n_epochs=30, patience=10):
    global optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_val_auroc, epochs_no_improve = -float('inf'), 0
    ckpt_path = f'/kaggle/working/{model_name}_best.pt'

    for epoch in range(1, n_epochs + 1):
        train_m = run_epoch_generic(model, train_loader, dual_input, train=True)
        val_m = run_epoch_generic(model, val_loader, dual_input, train=False)
        val_cls = compute_classification_metrics(val_m['labels'], val_m['preds'])
        print(f"[{model_name}] Epoch {epoch:2d}/{n_epochs} | train_loss={train_m['loss']:.4f} | "
              f"val_loss={val_m['loss']:.4f} val_auroc={val_cls['auroc']:.3f}")

        if val_cls['auroc'] > best_val_auroc:
            best_val_auroc = val_cls['auroc']
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{model_name}] Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(ckpt_path))
    test_m = run_epoch_generic(model, test_loader, dual_input, train=False)
    test_cls = compute_classification_metrics(test_m['labels'], test_m['preds'])
    test_reg = compute_regression_metrics(test_m['reg_true'], test_m['reg_preds'], severity_mean, severity_std)

    df = pd.DataFrame({'subject': subj_test, 'true_label': test_m['labels'], 'pred_prob': test_m['preds']})
    subj_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
    subj_auroc = roc_auc_score(subj_level['true_label'], subj_level['mean_pred_prob'])

    print(f"\n[{model_name}] TEST — Window AUROC: {test_cls['auroc']:.4f} | Subject AUROC: {subj_auroc:.4f} | "
          f"F1: {test_cls['f1']:.4f} | MAE: {test_reg['mae']:.3f}\n")

    return {'model': model_name, 'window_auroc': test_cls['auroc'], 'subject_auroc': subj_auroc,
            'f1': test_cls['f1'], 'mae': test_reg['mae'], 'rmse': test_reg['rmse']}


# ============================================================
# 5. RUN ALL THREE EXPERIMENTS
# ============================================================

BATCH_SIZE = 64
results_all = []

# --- Experiment A: IMU (3 sensors) only ---
train_ds_a = SingleModalityDataset(X_imu_train_small, y_train, sev_train_scaled, imu_mean_small, imu_std_small, augment=True)
val_ds_a = SingleModalityDataset(X_imu_val_small, y_val, sev_val_scaled, imu_mean_small, imu_std_small, augment=False)
test_ds_a = SingleModalityDataset(X_imu_test_small, y_test, sev_test_scaled, imu_mean_small, imu_std_small, augment=False)
loaders_a = (DataLoader(train_ds_a, BATCH_SIZE, shuffle=True), DataLoader(val_ds_a, BATCH_SIZE), DataLoader(test_ds_a, BATCH_SIZE))
model_a = SingleModalityModel(X_imu_train_small.shape[2]).to(device)
results_all.append(train_and_evaluate(model_a, "3sensor_IMU_only", *loaders_a, dual_input=False))

# --- Experiment B: Naive fusion (3-sensor IMU + insole) ---
train_ds_b = DualModalityDataset(X_imu_train_small, X_insole_train, y_train, sev_train_scaled, imu_mean_small, imu_std_small, augment=True)
val_ds_b = DualModalityDataset(X_imu_val_small, X_insole_val, y_val, sev_val_scaled, imu_mean_small, imu_std_small, augment=False)
test_ds_b = DualModalityDataset(X_imu_test_small, X_insole_test, y_test, sev_test_scaled, imu_mean_small, imu_std_small, augment=False)
loaders_b = (DataLoader(train_ds_b, BATCH_SIZE, shuffle=True), DataLoader(val_ds_b, BATCH_SIZE), DataLoader(test_ds_b, BATCH_SIZE))
model_b = NaiveLateFusionModel(X_imu_train_small.shape[2], X_insole_train.shape[2]).to(device)
results_all.append(train_and_evaluate(model_b, "3sensor_naive_fusion", *loaders_b, dual_input=True))

# --- Experiment C: Cross-attention fusion (3-sensor IMU + insole) ---
model_c = GaitFuseNetModel(X_imu_train_small.shape[2], X_insole_train.shape[2]).to(device)
results_all.append(train_and_evaluate(model_c, "3sensor_cross_attention", *loaders_b, dual_input=True))  # reuse loaders_b, same data

# ============================================================
# 6. FINAL COMPARISON
# ============================================================

results_df = pd.DataFrame(results_all)
print("\n" + "="*70)
print("3-SENSOR REDUCTION EXPERIMENT — SUMMARY")
print("="*70)
print(results_df.to_string(index=False))
results_df.to_csv('/kaggle/working/sensor_reduction_results.csv', index=False)

Keeping 66 IMU columns (3 sensors x 22 each): ['LowerBack', 'R_Ankle', 'L_Ankle']
Reduced IMU shapes: (17066, 300, 66) (4006, 300, 66) (3493, 300, 66)
[3sensor_IMU_only] Epoch  1/30 | train_loss=0.9638 | val_loss=1.2253 val_auroc=0.954
[3sensor_IMU_only] Epoch  2/30 | train_loss=0.6988 | val_loss=1.4930 val_auroc=0.947
[3sensor_IMU_only] Epoch  3/30 | train_loss=0.6229 | val_loss=1.2945 val_auroc=0.959
[3sensor_IMU_only] Epoch  4/30 | train_loss=0.5545 | val_loss=1.0690 val_auroc=0.972
[3sensor_IMU_only] Epoch  5/30 | train_loss=0.5106 | val_loss=1.1653 val_auroc=0.953
[3sensor_IMU_only] Epoch  6/30 | train_loss=0.4769 | val_loss=1.3221 val_auroc=0.953
[3sensor_IMU_only] Epoch  7/30 | train_loss=0.4424 | val_loss=1.2728 val_auroc=0.945
[3sensor_IMU_only] Epoch  8/30 | train_loss=0.4083 | val_loss=1.2989 val_auroc=0.959
[3sensor_IMU_only] Epoch  9/30 | train_loss=0.3666 | val_loss=1.2430 val_auroc=0.962
[3sensor_IMU_only] Epoch 10/30 | train_loss=0.3512 | val_loss=1.2294 val_auroc=0.955